In [8]:
from google.colab import drive
drive.mount('/content/drive')

import joblib
import pandas as pd
import numpy as np
import requests # Import the requests library

# Change this path to wherever you saved the files in Drive
MODEL_URL = 'https://drive.google.com/uc?export=download&id=1kvspp5sdaO8Ks_AjQ4IIFWy4MnxpEcbg'
SCALER_URL = 'https://drive.google.com/uc?export=download&id=166zJHvBnH74TzrlVB6Rxy8mfz-D_NSVv'

# Define local paths for the downloaded files
MODEL_PATH = '/tmp/best_model.pkl'
SCALER_PATH = '/tmp/scaler.pkl'

# Download the model file
print(f"Downloading model from {MODEL_URL}...")
response = requests.get(MODEL_URL)
response.raise_for_status() # Raise an exception for HTTP errors
with open(MODEL_PATH, 'wb') as f:
    f.write(response.content)
print("Model downloaded successfully.")

# Download the scaler file
print(f"Downloading scaler from {SCALER_URL}...")
response = requests.get(SCALER_URL)
response.raise_for_status() # Raise an exception for HTTP errors
with open(SCALER_PATH, 'wb') as f:
    f.write(response.content)
print("Scaler downloaded successfully.")

model = joblib.load(MODEL_PATH)
scaler = joblib.load(SCALER_PATH)

# Must match training feature order exactly
feature_names = ['Irms', 'Ax', 'Ay', 'Az', 'vibration_mag', 'vibration_delta', 'vibration_std']

# Example new reading
new_sample = pd.DataFrame([
    {
        'Irms': 0.50,
        'Ax': 9.75,
        'Ay': 0.20,
        'Az': 1.10,
        'vibration_mag': 9.82,
        'vibration_delta': 0.05,
        'vibration_std': 0.02
    }
])

X_new = new_sample[feature_names]
X_new_scaled = scaler.transform(X_new)

prediction = model.predict(X_new_scaled)[0]

if hasattr(model, "predict_proba"):
    probs = model.predict_proba(X_new_scaled)[0]
else:
    probs = None

label_map = {
    0: "Normal",
    1: "Fault Type 1",
    2: "Fault Type 2",
    3: "Fault Type 3"
}

print("Predicted label:", prediction)
print("Predicted state:", label_map.get(int(prediction), "Unknown"))

if probs is not None:
    print("Class probabilities:", probs)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model downloaded successfully.
Scaler downloaded successfully.
Predicted label: 3
Predicted state: Fault Type 3
Class probabilities: [0.00666667 0.13333333 0.29       0.57      ]
